# Metrics for rankings & search results

_Metrics & Evaluation — measuring models, data & statistics_

**A correct answer at rank 1 beats the same answer at rank 10 — ranking metrics reward good ORDER, not just the right set.**

Start with the everyday picture. You search for a recipe. The engine returns 10 links. You glance at the top one or two, maybe scroll a little, then either click or give up. You almost never reach the bottom. So a ranker that buries the perfect recipe at position 9 has failed you — even though, as a set, it "found" the right page.

       That is the core insight: a ranked list is judged top-down, with attention fading as you go down. A good ranking metric must therefore (1) reward putting relevant items high, and (2) discount items that sit lower. "Relevant" can be binary (relevant / not) or graded (a 0–3 score of how good a match is). The whole family below is built from those two ingredients — a notion of relevance, and a way to weight by position.

---

This notebook is a practice scaffold for the **Metrics for rankings & search results** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.metrics import ndcg_score, label_ranking_average_precision_score
from scipy.stats import kendalltau, spearmanr

# --- one query: 5 results, scored by the model, with TRUE relevance grades ---
# graded relevance 0..3 (0 = useless, 3 = perfect)
true_rel = np.array([[3, 2, 0, 0, 1]])     # ground-truth grade of each item
scores   = np.array([[0.9, 0.2, 0.7, 0.1, 0.5]])  # model's score for each item

# --- NDCG@k: position-weighted, graded relevance (the search/recsys workhorse) ---
print("NDCG@3 :", round(ndcg_score(true_rel, scores, k=3), 4))
print("NDCG@5 :", round(ndcg_score(true_rel, scores, k=5), 4))

# --- DCG from scratch, to demystify ndcg_score ---
def dcg_at_k(rel_in_ranked_order, k):
    g = np.asarray(rel_in_ranked_order)[:k]
    discounts = 1.0 / np.log2(np.arange(2, g.size + 2))
    return float(np.sum(g * discounts))
order = np.argsort(scores[0])[::-1]        # rank items best-first by score
rel_ranked = true_rel[0][order]
idcg = dcg_at_k(np.sort(true_rel[0])[::-1], 5)
print("DCG@5  :", round(dcg_at_k(rel_ranked, 5), 4),
      " NDCG@5 (manual):", round(dcg_at_k(rel_ranked, 5) / idcg, 4))

# --- LRAP: sklearn's label-ranking average precision (binary relevance) ---
bin_rel = (true_rel > 0).astype(int)       # 1 = relevant, 0 = not
print("LRAP   :", round(label_ranking_average_precision_score(bin_rel, scores), 4))

# --- from-scratch Precision@k, MRR, MAP on binary relevance ---
def precision_at_k(rel_ranked, k):
    return float(np.asarray(rel_ranked)[:k].sum()) / k

def reciprocal_rank(rel_ranked):
    hits = np.flatnonzero(np.asarray(rel_ranked) == 1)
    return 1.0 / (hits[0] + 1) if hits.size else 0.0   # rank of FIRST hit

def average_precision(rel_ranked):
    rel_ranked = np.asarray(rel_ranked)
    total = rel_ranked.sum()
    if total == 0:
        return 0.0
    hits, s = 0, 0.0
    for r, rr in enumerate(rel_ranked, start=1):
        if rr == 1:
            hits += 1
            s += hits / r                  # precision at this hit's rank
    return s / total

bin_ranked = bin_rel[0][order]
print("P@3    :", round(precision_at_k(bin_ranked, 3), 4))
print("RR     :", round(reciprocal_rank(bin_ranked), 4))
print("AP     :", round(average_precision(bin_ranked), 4))

# MRR / MAP are these averaged over many queries:
queries = [bin_ranked, np.array([1, 0, 1, 0, 0]), np.array([0, 0, 0, 1, 1])]
print("MRR    :", round(np.mean([reciprocal_rank(q) for q in queries]), 4))
print("MAP    :", round(np.mean([average_precision(q) for q in queries]), 4))

# --- rank correlation: do two whole orderings agree? ---
gold_order  = [1, 2, 3, 4, 5]              # ideal positions
model_order = [1, 3, 2, 5, 4]              # model's positions
print("Kendall tau :", round(kendalltau(gold_order, model_order).statistic, 4))
print("Spearman rho:", round(spearmanr(gold_order, model_order).statistic, 4))

## Visualize the data & results

_On real data: rank the breast-cancer test tumors by a (deliberately weak) score and treat truly-malignant tumors as 'relevant'. How do Precision@k and NDCG@k change as we look further down the list (grow k)?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

# relevant = malignant tumor (sklearn target: 0=malignant, 1=benign)
data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)          # 1 = malignant = "relevant"
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.4, random_state=0, stratify=y)

# a DELIBERATELY WEAK ranker: rank tumors by ONE noisy feature (mean texture)
# so the ranking is imperfect and the curves actually have structure.
scores = X_te[:, 1].astype(float)           # feature 1 = "mean texture"
order  = np.argsort(scores)[::-1]           # rank best-first
rel    = y_te[order]                        # 1/0 relevance in ranked order
R      = int(rel.sum())                     # total relevant (= 85)

def precision_at_k(rel, k): return rel[:k].sum() / k
def recall_at_k(rel, k):    return rel[:k].sum() / R
def dcg_at_k(rel, k):
    g = rel[:k]; disc = 1.0 / np.log2(np.arange(2, k + 2)); return float(np.sum(g * disc))
def ndcg_at_k(rel, k):
    idcg = dcg_at_k(np.sort(rel)[::-1], k)
    return dcg_at_k(rel, k) / idcg if idcg > 0 else 0.0

ks = [1, 2, 3, 5, 8, 10, 15, 20, 30, 40, 50]
print("k  P@k    R@k    NDCG@k")
for k in ks:
    print(k, round(precision_at_k(rel, k), 3),
             round(recall_at_k(rel, k), 3),
             round(ndcg_at_k(rel, k), 3))

# cross-check NDCG against scikit-learn's ndcg_score
sorted_scores = np.sort(scores)[::-1].reshape(1, -1)
y_true_row    = rel.reshape(1, -1).astype(float)
for k in [5, 10, 20]:
    print("sklearn NDCG@%d:" % k, round(ndcg_score(y_true_row, sorted_scores, k=k), 3))
# -> 0.854, 0.696, 0.666  (matches the from-scratch values above)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Two rankers return the SAME 5 documents and find the SAME 2 relevant ones, so both have Recall@5 = 1.0. Ranker A places the relevant docs at ranks 1 and 2; Ranker B places them at ranks 4 and 5. A stakeholder says "they're tied — both found everything." Are they tied? Which single metric best shows the difference, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice what Recall@5 ignores. — _Recall@k only asks whether the relevant items are somewhere in the top $k$ — it says nothing about WHERE in the top $k$. Both rankers captured both relevant docs, so both score 1.0._
- Bring in position. — _Users read top-down, so ranks 1–2 (Ranker A) are far better than ranks 4–5 (Ranker B). We need a metric that discounts lower positions._
- Pick NDCG@5. — _NDCG weights each relevant item by $1/\log_2(\text{rank}+1)$ and normalizes by the ideal. Ranker A is the ideal ordering, so NDCG = 1.0; Ranker B is penalized for the deep placements, scoring well below 1.0._

**Answer:** Not tied. Recall@5 is identical (1.0 for both) precisely because it is blind to order. NDCG@5 best exposes the gap: Ranker A places both relevant docs at ranks 1–2, which is the ideal ordering, so NDCG@5 = 1.0; Ranker B's docs at ranks 4–5 get discounted by $1/\log_2 5 \approx 0.43$ and $1/\log_2 6 \approx 0.39$, dragging its NDCG well below 1.0. (MRR would also separate them — A's first hit is rank 1 ⇒ 1.0, B's is rank 4 ⇒ 0.25 — but MRR only looks at the FIRST hit, whereas NDCG accounts for both relevant items.)

</details>

**Problem 2.** Your search log has two query types: "navigational" queries where the user wants exactly ONE specific page (their bank's login), and "research" queries where the user wants to gather MANY relevant articles. You can only headline one metric per type. Which metric for each, and why not use the same one for both?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Characterize the navigational query. — _Success means the one intended page is at the very top; everything else is irrelevant. Only the rank of that first (and only) correct result matters._
- Pick MRR for navigational. — _MRR = $1/\text{rank}$ of the first relevant hit. It is exactly "how high did we put the one right answer", which is the entire goal for navigational queries._
- Characterize the research query. — _Here there are many relevant articles and the user wants them all surfaced high — getting one right is not enough._
- Pick MAP (or NDCG) for research. — _MAP rewards pulling ALL relevant items toward the top (it re-measures precision at each relevant item). NDCG works too, and is preferred if relevance is graded rather than binary._

**Answer:** Navigational ⇒ MRR. Only one page is correct and only its rank matters, so the reciprocal rank of the first (and only) hit is the right scoreboard. Research ⇒ MAP (or NDCG@k for graded relevance). The user wants every relevant article surfaced high, and MAP averages precision over all relevant items, rewarding rankings that cluster them near the top. Using MRR for research would ignore everything after the first hit; using MAP for navigational is overkill (and noisier) when there is exactly one correct answer.

</details>

**Problem 3.** A teammate builds an offline eval: for each of 1,000 queries they compute Recall@10 and average them. Query set A has ~3 relevant docs per query; query set B has ~40 relevant docs per query. The ranker scores great on A but "terribly" on B, and they conclude the ranker is broken for B-type queries. What is the flaw, and what should they report instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the ceiling on Recall@10 for each set. — _For B, at most 10 relevant docs can fit in the top 10, but there are ~40 relevant, so Recall@10 is capped at 10/40 = 0.25 no matter how perfect the ranker is. For A (3 relevant) it can reach 1.0._
- Spot the unfair comparison. — _The metric's maximum differs across query sets, so averaging Recall@10 penalizes B for having more relevant items — an artifact, not a ranker failure._
- Switch to a normalized or per-query-sized metric. — _NDCG normalizes by each query's ideal (best-possible = 1.0 for every query), so it is comparable across sets. R-precision (recall at each query's own number of relevant items) also removes the ceiling artifact._

**Answer:** The flaw is averaging a recall@k whose ceiling depends on the number of relevant items. B-type queries have ~40 relevant docs, so Recall@10 can never exceed 10/40 = 0.25 even for a perfect ranker — the low score is an artifact of the metric, not the model. Report NDCG@k instead (it normalizes by each query's ideal DCG, so a perfect ranking is 1.0 for every query regardless of how many relevant items it has), or use R-precision (recall at each query's own number of relevant items). Then A and B are on the same footing and the comparison is meaningful.

</details>